# Divar Real Estate Analysis — Recommender System (Clustering)

**Problem 1:** Clustering properties to support a property recommender system

**Author:** Mahdi Tarrah

**Data:** `data/Divar.csv`

In [1]:
#libraries
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import utm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
clustering_columns = ['price_value','building_size','rooms_count','location_latitude','location_longitude','cat2_slug','cat3_slug','city_slug','construction_year', 'has_parking', 'has_elevator']
df = pd.read_csv('../data/Divar.csv', usecols=clustering_columns)
print(df.shape)
df.head()

(1000000, 11)


,cat2_slug,cat3_slug,city_slug,price_value,building_size,rooms_count,has_elevator,has_parking,construction_year,location_latitude,location_longitude
0,temporary-rent,villa,karaj,NaN,500.0,سه,NaN,NaN,NaN,35.811684,50.936600
1,residential-sell,apartment-sell,tehran,8.500000e+09,60.0,یک,True,True,۱۳۸۴,NaN,NaN
2,residential-rent,apartment-rent,tehran,NaN,132.0,سه,True,True,۱۴۰۱,35.703865,51.373459
3,commercial-rent,office-rent,tehran,NaN,90.0,یک,True,True,۱۴۰۰,NaN,NaN
4,residential-sell,apartment-sell,mashhad,5.750000e+09,115.0,دو,True,True,۱۴۰۳,NaN,NaN


In [3]:
#filled
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   cat2_slug           1000000 non-null  str    
 1   cat3_slug           999999 non-null   str    
 2   city_slug           999998 non-null   str    
 3   price_value         568346 non-null   float64
 4   building_size       980394 non-null   float64
 5   rooms_count         845899 non-null   str    
 6   has_elevator        541749 non-null   object 
 7   has_parking         728156 non-null   object 
 8   construction_year   815828 non-null   str    
 9   location_latitude   655608 non-null   float64
 10  location_longitude  655608 non-null   float64
dtypes: float64(4), object(2), str(5)
memory usage: 83.9+ MB


In [4]:
#copy
data = df.copy()

#clean
data.loc[data['price_value'] <= 0, 'price_value'] = np.nan

def clean_outliers(data,column):
    """Set outliers to the NaN."""
    logs = np.log(data[column])
    q1, q3 = logs.quantile(0.25),logs.quantile(0.75)
    iqr = q3 - q1
    data.loc[~logs.between(q1 - 1.5 * iqr,q3 + 1.5 * iqr), column] = np.nan
    return data

data = clean_outliers(data,'price_value')
data = clean_outliers(data,'building_size')

inside_iran = (data['location_latitude'].between(25,40)) & (data['location_longitude'].between(44,63.5))
data.loc[~inside_iran,['location_latitude','location_longitude']] = np.nan

data = data.dropna(subset=['building_size','price_value','location_latitude','location_longitude'])
print(data.shape)
print(data[['price_value','building_size']].describe())

(324485, 11)
        price_value  building_size
count  3.244850e+05  324485.000000
mean   5.047053e+09     130.622852
std    6.172471e+09      87.705984
min    1.640000e+08      23.000000
25%    1.600000e+09      75.000000
50%    3.000000e+09     100.000000
75%    5.850000e+09     153.000000
max    5.100000e+10     538.000000


In [5]:
xy = [utm.from_latlon(lat, lon, force_zone_number=39)[:2] 
for lat, lon in zip(data['location_latitude'], data['location_longitude'])]

data['utm_x'] = [p[0] for p in xy]
data['utm_y'] = [p[1] for p in xy]

print(data[['utm_x','utm_y']].describe())

              utm_x         utm_y
count  3.244850e+05  3.244850e+05
mean   5.549497e+05  3.888984e+06
std    2.790465e+05  2.525020e+05
min   -1.146343e+05  2.803746e+06
25%    4.763461e+05  3.860848e+06
50%    5.304762e+05  3.953651e+06
75%    5.738981e+05  4.042502e+06
max    1.660176e+06  4.407134e+06


In [6]:
#cheak errors
print(((data['utm_x'] < 100000) | (data['utm_x'] > 900000)).sum())

51619


In [7]:
#rooms
rooms_mapping = {'بدون اتاق':0,'یک':1,'دو':2,'سه':3,'چهار':4,'پنج یا بیشتر':5}
data['rooms_count'] = data['rooms_count'].map(rooms_mapping)

In [8]:
print(data[['construction_year','has_parking','has_elevator']].info())
print(data['construction_year'].value_counts(dropna=False).head(10))

<class 'pandas.DataFrame'>
Index: 324485 entries, 7 to 999995
Data columns (total 3 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   construction_year  275148 non-null  str   
 1   has_parking        266716 non-null  object
 2   has_elevator       200577 non-null  object
dtypes: object(2), str(1)
memory usage: 9.9+ MB
None
construction_year
۱۴۰۳    50587
NaN     49337
۱۴۰۲    22978
۱۴۰۰    15228
۱۳۹۰    14977
۱۳۹۵    14602
۱۴۰۱    12499
۱۳۹۷    11304
۱۳۹۸    11174
۱۳۹۶    11151
Name: count, dtype: int64


In [9]:
#convert to number
digits = str.maketrans('۰۱۲۳۴۵۶۷۸۹', '0123456789')
year = data['construction_year'].replace('قبل از ۱۳۷۰', '۱۳۶۹')
data['construction_year'] = pd.to_numeric(year.str.translate(digits), errors='coerce')
print(data['construction_year'].describe())

count    275148.000000
mean       1394.260892
std           8.411427
min        1369.000000
25%        1390.000000
50%        1396.000000
75%        1402.000000
max        1403.000000
Name: construction_year, dtype: float64


In [10]:
data_cmp = data.dropna(subset=['rooms_count','construction_year'])
print(data.shape, data_cmp.shape)

(324485, 13) (275143, 13)


In [11]:
print(data_cmp[['price_value','building_size','rooms_count','construction_year']].corr())

                   price_value  building_size  rooms_count  construction_year
price_value           1.000000       0.398886     0.344050          -0.026310
building_size         0.398886       1.000000     0.617901           0.090851
rooms_count           0.344050       0.617901     1.000000           0.116831
construction_year    -0.026310       0.090851     0.116831           1.000000


In [12]:
feature_sets = {'A: price + utm + size': ['price_value','utm_x','utm_y','building_size'],'B: A + rooms':['price_value','utm_x','utm_y','building_size','rooms_count'],'C: price + utm':['price_value','utm_x','utm_y'],'D: C + year': ['price_value','utm_x','utm_y','construction_year']}

for name, feats in feature_sets.items():
    X = StandardScaler().fit_transform(data_cmp[feats])
    km = KMeans(10, random_state=44)
    labels = km.fit_predict(X)
    print(name)
    print(silhouette_score(X, labels, sample_size=10000, random_state=44))
    print(calinski_harabasz_score(X, labels))
    print(davies_bouldin_score(X, labels))

A: price + utm + size
0.38103529189277213
102768.06431253256
1.0292128341314926
B: A + rooms
0.34443408378687007
74908.88484914752
1.0877651052155721
C: price + utm
0.4757394657372042
186268.2007138129
0.75140171115489
D: C + year
0.329410603147055
94459.92883366671
0.9917509071346811
